In [30]:
import itertools
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
import neat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.special import softmax
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import unicodedata
import re
import configparser
from sklearn.decomposition import TruncatedSVD


In [17]:
SEED = 42

np.random.seed(SEED)


In [18]:
_STOPWORDS = stopwords.words("spanish")  # agregar más palabras a esta lista si es necesario
PUNCTUACTION = ";:,.\\-\"'/"
SYMBOLS = "()[]¿?¡!{}~<>|@#"
NUMBERS= "0123456789"
SKIP_SYMBOLS = set(PUNCTUACTION + SYMBOLS)
SKIP_SYMBOLS_AND_SPACES = set(PUNCTUACTION + SYMBOLS + '\t\n\r ')

def normaliza_texto(input_str,
                    punct=False,
                    accents=False,
                    num=False,
                    max_dup=2):
    """
        punct=False (elimina la puntuación, True deja intacta la puntuación)
        accents=False (elimina los acentos, True deja intactos los acentos)
        num= False (elimina los números, True deja intactos los acentos)
        max_dup=2 (número máximo de símbolos duplicados de forma consecutiva, rrrrr => rr)
    """
    
    nfkd_f = unicodedata.normalize('NFKD', input_str)
    n_str = []
    c_prev = ''
    cc_prev = 0
    for c in nfkd_f:
        if not num:
            if c in NUMBERS:
                continue
        if not punct:
            if c in SKIP_SYMBOLS:
                continue
        if not accents and unicodedata.combining(c):
            continue
        if c_prev == c:
            cc_prev += 1
            if cc_prev >= max_dup:
                continue
        else:
            cc_prev = 0
        n_str.append(c)
        c_prev = c
    texto = unicodedata.normalize('NFKD', "".join(n_str))
    texto = re.sub(r'(\s)+', r' ', texto.strip(), flags=re.IGNORECASE)
    return texto

def mi_preprocesamiento(texto):
    #convierte a minúsculas el texto antes de normalizar
    # print("antes: ", texto)
    texto = normaliza_texto(texto.lower())
    # print("después:",texto)
    return texto

def mi_tokenizador(texto):
    # Elimina stopwords: palabras que no se consideran de contenido y que no agregan valor semántico al texto
    #print("antes: ", texto)
    texto = [t for t in texto.split() if t not in _STOPWORDS]
    #print("después:",texto)
    return texto

In [31]:
def update_neat_config(config_path, cambios):

    conf = configparser.ConfigParser()
    conf.read(config_path)

    for param, valor in cambios.items():

        if param == 'pop_size':
            conf.set('NEAT', param, str(valor))

        elif param in [
            'num_inputs',
            'num_outputs',
            'node_add_prob',
            'conn_add_prob',
            'conn_delete_prob',
            'weight_mutate_rate'
        ]:
            conf.set('DefaultGenome', param, str(valor))

        elif param == 'compatibility_threshold':
            conf.set('DefaultSpeciesSet', param, str(valor))

        elif param == 'survival_threshold':
            conf.set('DefaultReproduction', param, str(valor))

    temp_config = "temp_config.cfg"

    with open(temp_config, "w") as f:
        conf.write(f)

    return neat.Config(
        neat.DefaultGenome,
        neat.DefaultReproduction,
        neat.DefaultSpeciesSet,
        neat.DefaultStagnation,
        temp_config
    )

In [20]:
def make_eval_function(X_train, Y_train_encoded):
    def eval_genomes(genomes, config):

        batch_size = min(512, len(X_train))

        indices = np.random.choice(len(X_train), batch_size, replace=False)

        X_batch = X_train[indices]
        Y_batch = Y_train_encoded[indices]

        for genome_id, genome in genomes:

            net = neat.nn.FeedForwardNetwork.create(genome, config)

            predictions = []

            for xi in X_batch:
                output = net.activate(xi)
                pred = np.argmax(output)
                predictions.append(pred)

            fitness = f1_score(
                Y_batch,
                predictions,
                average="macro"
            )
            complexity_penalty = len(genome.connections) * 0.0005

            genome.fitness = fitness - complexity_penalty
    return eval_genomes

In [21]:
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)

In [22]:
le = LabelEncoder()

In [ ]:
experimentos = {
    'pop_size': [100],
    'node_add_prob': [0.1],
    'conn_add_prob': [0.1],
    'conn_delete_prob': [0.1],
    'weight_mutate_rate': [0.9],
    'compatibility_threshold': [2.5],
    'survival_threshold': [0.5]
}

# Representaciones a probar
representaciones = ['TFIDF', 'WE']

dimensiones = [0.20, 0.25, 0.30]

resultados_totales = []

combinaciones = list(itertools.product(*experimentos.values()))

for repr_type in representaciones:
    for dim_pct in dimensiones:
        # 1. Preparar datos según representación
        if repr_type == 'TFIDF':
            X_train = dataset_train["text"].to_numpy()
            Y_train = dataset_train["klass"].to_numpy()
            X_test = dataset_test["text"].to_numpy()
            Y_test = dataset_test["klass"].to_numpy()
            Y_encoded= le.fit_transform(Y_train)
            Y_test_encoded = le.transform(Y_test) 

            vec_tfidf = CountVectorizer(analyzer="word", preprocessor=mi_preprocesamiento, tokenizer=mi_tokenizador,  ngram_range=(1,1), max_features=3000, min_df=2, max_df=0.9)
            X_tfidf = vec_tfidf.fit_transform(X_train)
            X_test_tfidf = vec_tfidf.transform(X_test)  

            total_features = X_tfidf.shape[1] 
            n_comp = int(total_features * dim_pct)
            svd = TruncatedSVD(n_components=n_comp, random_state=42)

            X_reduced = svd.fit_transform(X_tfidf)
            X_test = svd.transform(X_test_tfidf)

            X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
                X_reduced,
                Y_encoded,
                test_size=0.2,
                random_state=42,
                stratify=Y_encoded
            )

        else:
            X_trainext = dataset_train['text'].to_numpy()
            # Convertir a matriz de arrays de numpy
            X_es = np.vstack(dataset_train["we_es"].to_numpy())
            X_mx = np.vstack(dataset_train["we_mx"].to_numpy())
            X_ft = np.vstack(dataset_train["we_ft"].to_numpy())
            Y_train = dataset_train['klass'].to_numpy()

            X_train = np.concatenate([X_es, X_mx, X_ft], axis=1) 
            
            X_test_text = dataset_test['text'].to_numpy()
            # Convertir a matriz de arrays de numpy
            X_es_t = np.vstack(dataset_test["we_es"].to_numpy())
            X_mx_t = np.vstack(dataset_test["we_mx"].to_numpy())
            X_ft_t = np.vstack(dataset_test["we_ft"].to_numpy())
            X_test = np.concatenate([X_es_t, X_mx_t, X_ft_t], axis=1) 
            Y_test = dataset_test['klass'].to_numpy()

            scaler = StandardScaler()

            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            Y_train_encoded= le.fit_transform(Y_train)
            Y_test_encoded= le.transform(Y_test)

            total_features = X_train.shape[1] # 900
            n_comp = int(total_features * dim_pct)
            pca = PCA(n_components=n_comp)

            X_train = pca.fit_transform(X_train)
            X_test = pca.transform(X_test)


            X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
                X_train,
                Y_train_encoded,
                test_size=0.2,
                random_state=42,
                stratify=Y_train_encoded
            )

        input_size = X_train.shape[1]

        for params in combinaciones:
            # Mapear valores del producto a nombres de parámetros
            dict_cambios = dict(zip(experimentos.keys(), params))
            dict_cambios['num_inputs'] = input_size # Ajuste automático de entrada 
            
            print(f"Probando: {repr_type} | Params: {dict_cambios}, Total: {len(combinaciones)}")

            # 2. Configurar NEAT
            config_actual = update_neat_config('config-feedforward.cfg', dict_cambios)
            
            # 3. Ejecutar Evolución
            p = neat.Population(config_actual)
            # p.add_reporter(neat.StdOutReporter(False))
            
            eval_function = make_eval_function(
                X_train,
                Y_train_encoded
            )

            winner = p.run(eval_function, 150)

            # 4. Evaluación en TEST 
            winner_net = neat.nn.FeedForwardNetwork.create(winner, config_actual)
            test_preds = [np.argmax(winner_net.activate(xi)) for xi in X_test]
            f1_test = f1_score(Y_test_encoded, test_preds, average="macro")

            # 5. Registrar 
            resultados_totales.append({
                'Representacion': repr_type,
                'Fitness_Evolutivo': winner.fitness,
                'F1_Test': f1_test,
                'Nodos_Finales': len(winner.nodes),
                'Conexiones_Finales': len(winner.connections),
                **dict_cambios
            })

# Guardar resultados
df = pd.DataFrame(resultados_totales)
df.to_csv("reporte_experimentos_neat.csv", index=False)